# Step 1: Persona Induction & Validation

This notebook induces distinct personas in the model and validates that they produce
distinguishable activation signatures. We test three induction methods:

1. **System prompt** — different framing instructions
2. **Few-shot examples** — demonstrations of target behaviour
3. **Activation injection** — direct activation-space offsets (optional)

We validate by collecting mean activations across a set of neutral prompts and measuring
per-layer cosine similarity between persona pairs.

In [ ]:
import torch
from nnsight import LanguageModel

from persona_steering.config import GEMMA_2_9B, ExperimentConfig, TRAIT_CONFIGS, Trait
from persona_steering.personas import PersonaInducer, load_all_personas
from persona_steering.utils import get_device, ensure_output_dirs

ensure_output_dirs()
device = get_device()
print(f"Device: {device}")

In [ ]:
# Load model
config = GEMMA_2_9B
model = LanguageModel(config.hf_id, device_map="auto", torch_dtype=torch.float16)
tokenizer = model.tokenizer
print(f"Loaded {config.name}")

In [ ]:
# Load personas
personas = load_all_personas()
for p in personas:
    print(f"  {p.name}: {p.description[:80]}...")

In [ ]:
# Collect activations for each persona
inducer = PersonaInducer(model, tokenizer)
layers = config.extraction_layers

neutral_prompts = [
    "What is the capital of France?",
    "Explain how photosynthesis works.",
    "What are the benefits of exercise?",
    "Describe the water cycle.",
    "How does a computer processor work?",
    "What causes seasons on Earth?",
    "Explain supply and demand.",
    "What is the scientific method?",
    "How do vaccines work?",
    "Describe the structure of DNA.",
]

for persona in personas:
    inducer.collect_activations(persona, neutral_prompts, layers)

In [ ]:
# Validate persona distinctness
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [(personas[0], personas[1]), (personas[0], personas[2]), (personas[1], personas[2])]
for ax, (pa, pb) in zip(axes, pairs):
    sims = inducer.validate_persona(pa, pb, layers)
    layer_ids = sorted(sims.keys())
    sim_vals = [sims[l] for l in layer_ids]
    ax.plot(layer_ids, sim_vals, marker='o', markersize=3)
    ax.set_title(f"{pa.name} vs {pb.name}")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Cosine Similarity")
    ax.set_ylim(0.8, 1.0)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/persona_validation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Persona validation complete.")